# CFM interface-stiffness anisotropy: effect of 110[001] box thickness

Goal: add the thicker-box 110[001] data and compare the anisotropy-parameter distributions $\epsilon_1$ and $\epsilon_2$ obtained from different three-orientation combinations.

Note: this notebook excludes the 111 orientation, excludes the all-110 set, and never uses both 110[001] data sets simultaneously.

In [ ]:
from pathlib import Path
import os


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "interface_analyzer" / "reproducibility").exists():
            return path
    raise RuntimeError("Could not find repository root containing interface_analyzer/reproducibility")


PROJECT_ROOT = find_repo_root()
REPRO_DIR = PROJECT_ROOT / "interface_analyzer" / "reproducibility"
DATASET_DIR = REPRO_DIR / "dataset"
LOCAL_OUTPUT_DIR = REPRO_DIR / "_local_outputs"
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Full manuscript-scale post-processing files are intentionally not bundled.
# Set INTERFACE_ANALYZER_DATA to the directory containing those generated files.
FULL_DATA_ROOT = Path(os.environ.get("INTERFACE_ANALYZER_DATA", LOCAL_OUTPUT_DIR)).expanduser()
FULL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

# For quick local CFG tests this defaults to the bundled sample dataset.
CFG_DIR = Path(os.environ.get("INTERFACE_ANALYZER_CFG_DIR", DATASET_DIR)).expanduser()

print("Project root:", PROJECT_ROOT)
print("Bundled CFG dataset:", DATASET_DIR)
print("Analysis data root:", FULL_DATA_ROOT)
print("CFG input dir:", CFG_DIR)


## 1. Input Data

In [ ]:
import itertools
import os
from collections import OrderedDict

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import numpy as np


# Interface-stiffness data: three independent runs for each orientation; units follow the E-20 convention from the original notebook.
# The 111 orientation is excluded here, and the thicker-box 110[001] data are included.
STIFFNESS_SAMPLES_RAW = {
    "100[010]":  [86.3, 85.3, 83.7],
    "110[001]_thin_box":  [80.2, 79.9, 78.9],
    "110[001]_thick_box": [85.5, 86.1, 87.0],
    "110[1-10]": [121.7, 124.1, 122.7],
    "110[1-12]": [95.5, 93.9, 93.9],
}

UNIT_SCALE = 1e-20
STIFFNESS_SAMPLES = {
    name: np.array(values, dtype=float) * UNIT_SCALE
    for name, values in STIFFNESS_SAMPLES_RAW.items()
}

# beta = gamma0 * (c0 + c1*epsilon1 + c2*epsilon2)
STIFFNESS_MODELS = {
    "100[010]":  (1.0, -18/5,   -80/7),
    "110[001]_thin_box":  (1.0, -21/10, 365/14),
    "110[001]_thick_box": (1.0, -21/10, 365/14),
    "110[1-10]": (1.0,  39/10,  155/14),
    "110[1-12]": (1.0, -1/10,   295/14),
}

ORIENTATION_FAMILY = {
    "100[010]": "100[010]",
    "110[001]_thin_box": "110[001]",
    "110[001]_thick_box": "110[001]",
    "110[1-10]": "110[1-10]",
    "110[1-12]": "110[1-12]",
}

## 2. Computation functions

In [ ]:
def solve_from_combo(stiffness_dict, combo, models=STIFFNESS_MODELS):
    """Solve gamma0, epsilon1, and epsilon2 from stiffnesses in three directions."""
    A = np.array([models[name] for name in combo], dtype=float)
    b = np.array([stiffness_dict[name] for name in combo], dtype=float)

    # x = [gamma0, gamma0*epsilon1, gamma0*epsilon2]
    x1, x2, x3 = np.linalg.solve(A, b)
    gamma0 = x1
    epsilon1 = x2 / x1
    epsilon2 = x3 / x1

    return gamma0, epsilon1, epsilon2, np.linalg.cond(A)


def valid_direction_combos(samples=STIFFNESS_SAMPLES):
    """List all available three-direction combinations.

    Filtering rules:
    1. Do not use all three 110 directions.
    2. Do not use both 110[001] data sets at the same time.
    """
    names = list(samples)
    combos = []

    for combo in itertools.combinations(names, 3):
        families = [ORIENTATION_FAMILY[name] for name in combo]
        n_110 = sum(family.startswith("110") for family in families)

        if n_110 == 3:
            continue
        if families.count("110[001]") > 1:
            continue
        combos.append(combo)

    return combos


def solve_all_replicate_combinations(samples=STIFFNESS_SAMPLES):
    """Compute the 3^3 = 27 independent-run combinations for each direction set."""
    rows = []

    for combo in valid_direction_combos(samples):
        for sample_indices in itertools.product(range(3), repeat=3):
            stiffness_dict = {
                name: samples[name][sample_idx]
                for name, sample_idx in zip(combo, sample_indices)
            }
            gamma0, epsilon1, epsilon2, cond_A = solve_from_combo(stiffness_dict, combo)

            rows.append({
                "direction_combo": " + ".join(combo),
                "sample_indices": sample_indices,
                "gamma0": gamma0,
                "epsilon1": epsilon1,
                "epsilon2": epsilon2,
                "minus_epsilon2": -epsilon2,
                "cond_A": cond_A,
            })

    return rows


def summarize_by_direction_combo(results):
    """Return the mean and sample standard deviation from the 27 results for each direction set."""
    grouped = OrderedDict()

    for row in results:
        grouped.setdefault(row["direction_combo"], []).append(row)

    summary = []
    for direction_combo, rows in grouped.items():
        summary.append({
            "direction_combo": direction_combo,
            "n": len(rows),
            "gamma0_mean": np.mean([row["gamma0"] for row in rows]),
            "gamma0_std": np.std([row["gamma0"] for row in rows], ddof=1),
            "epsilon1_mean": np.mean([row["epsilon1"] for row in rows]),
            "epsilon1_std": np.std([row["epsilon1"] for row in rows], ddof=1),
            "epsilon2_mean": np.mean([row["epsilon2"] for row in rows]),
            "epsilon2_std": np.std([row["epsilon2"] for row in rows], ddof=1),
            "minus_epsilon2_mean": np.mean([row["minus_epsilon2"] for row in rows]),
            "minus_epsilon2_std": np.std([row["minus_epsilon2"] for row in rows], ddof=1),
            "cond_A": rows[0]["cond_A"],
        })

    return summary


def print_table(rows, columns):
    """Print a compact table without adding a pandas dependency."""
    def format_value(value):
        if isinstance(value, (float, np.floating)):
            return f"{value:.10e}"
        return str(value)

    formatted_rows = [
        [format_value(row[column]) for column in columns]
        for row in rows
    ]
    widths = [
        max(len(column), *(len(row[i]) for row in formatted_rows))
        for i, column in enumerate(columns)
    ]

    header = "  ".join(column.ljust(width) for column, width in zip(columns, widths))
    print(header)
    print("  ".join("-" * width for width in widths))
    for row in formatted_rows:
        print("  ".join(value.ljust(width) for value, width in zip(row, widths)))

## 3. Compute distributions for different combinations

In [ ]:
results = solve_all_replicate_combinations()
summary = summarize_by_direction_combo(results)

print(f"Number of direction combinations: {len(summary)}")
print(f"Total number of independent-run combinations: {len(results)}")

print_table(
    summary,
    [
        "direction_combo",
        "n",
        "epsilon1_mean",
        "epsilon1_std",
        "epsilon2_mean",
        "epsilon2_std",
        "cond_A",
    ],
)

## 4. Overall statistics

In [ ]:
overall_stats = []
for quantity in ["gamma0", "epsilon1", "epsilon2", "minus_epsilon2"]:
    values = np.array([row[quantity] for row in results], dtype=float)
    label = "-epsilon2" if quantity == "minus_epsilon2" else quantity
    overall_stats.append({
        "quantity": label,
        "mean": np.mean(values),
        "std": np.std(values, ddof=1),
    })

print_table(overall_stats, ["quantity", "mean", "std"])

## 5. $\epsilon_1$ and $-\epsilon_2$ scatter plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

colors = plt.cm.tab10(np.linspace(0, 1, len(summary)))

for color, row in zip(colors, summary):
    combo_points = [
        point for point in results
        if point["direction_combo"] == row["direction_combo"]
    ]

    ax.scatter(
        [point["minus_epsilon2"] for point in combo_points],
        [point["epsilon1"] for point in combo_points],
        s=18,
        alpha=0.22,
        color=color,
    )

    ax.errorbar(
        row["minus_epsilon2_mean"],
        row["epsilon1_mean"],
        xerr=row["minus_epsilon2_std"],
        yerr=row["epsilon1_std"],
        fmt="o",
        capsize=4,
        markersize=6,
        color=color,
        label=row["direction_combo"],
    )

x_min, x_max = ax.get_xlim()
x_line = np.linspace(x_min, x_max, 200)
ax.plot(
    x_line,
    (20 / 3) * x_line,
    "k--",
    linewidth=1.5,
    label=r"$\epsilon_1=-\frac{20}{3}\epsilon_2$",
)

ax.set_xlabel(r"$-\epsilon_2$")
ax.set_ylabel(r"$\epsilon_1$")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8, loc="best")
fig.tight_layout()

## 6. Write txt files

Write three files for convenient plotting in Origin:

- `epsilon_box_thickness_replicate_results.txt`：all independent-run combination results.
- `epsilon_box_thickness_summary_by_direction_combo.txt`: means and standard deviations for each direction combination.
- `epsilon_box_thickness_origin_errorbar_data.txt`：the most commonly used columns for scatter/error-bar plotting in Origin.

In [ ]:
def write_table_txt(path, rows, columns):
    """Write tab-delimited txt files for direct import into Origin."""
    with open(path, "w", encoding="utf-8") as f:
        f.write("\t".join(columns) + "\n")
        for row in rows:
            values = []
            for column in columns:
                value = row[column]
                if isinstance(value, tuple):
                    value = ",".join(str(item + 1) for item in value)
                elif isinstance(value, (float, np.floating)):
                    value = f"{value:.12e}"
                values.append(str(value))
            f.write("\t".join(values) + "\n")


replicate_columns = [
    "direction_combo",
    "sample_indices",
    "gamma0",
    "epsilon1",
    "epsilon2",
    "minus_epsilon2",
    "cond_A",
]

summary_columns = [
    "direction_combo",
    "n",
    "gamma0_mean",
    "gamma0_std",
    "epsilon1_mean",
    "epsilon1_std",
    "epsilon2_mean",
    "epsilon2_std",
    "minus_epsilon2_mean",
    "minus_epsilon2_std",
    "cond_A",
]

origin_rows = []
for row in summary:
    origin_rows.append({
        "direction_combo": row["direction_combo"],
        "x_minus_epsilon2_mean": row["minus_epsilon2_mean"],
        "xerr_minus_epsilon2_std": row["minus_epsilon2_std"],
        "y_epsilon1_mean": row["epsilon1_mean"],
        "yerr_epsilon1_std": row["epsilon1_std"],
    })

origin_columns = [
    "direction_combo",
    "x_minus_epsilon2_mean",
    "xerr_minus_epsilon2_std",
    "y_epsilon1_mean",
    "yerr_epsilon1_std",
]

write_table_txt("epsilon_box_thickness_replicate_results.txt", results, replicate_columns)
write_table_txt("epsilon_box_thickness_summary_by_direction_combo.txt", summary, summary_columns)
write_table_txt("epsilon_box_thickness_origin_errorbar_data.txt", origin_rows, origin_columns)

print("Wrote output files:")
print("epsilon_box_thickness_replicate_results.txt")
print("epsilon_box_thickness_summary_by_direction_combo.txt")
print("epsilon_box_thickness_origin_errorbar_data.txt")